<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Soluciones — Pandas: estructuras, tipos, memoria y selección eficiente**

Entramos en el bloque de **Pandas profesional**. El objetivo no es repasar `DataFrame` y `Series` desde cero, sino entender qué decisiones aparentemente pequeñas pueden afectar a:

- corrección;
- legibilidad;
- memoria;
- velocidad;
- reproducibilidad;
- mantenibilidad.

Seguiremos el patrón del curso:

> **observar → medir → detectar → corregir → volver a medir → interpretar**

Trabajaremos con un dataset suficientemente grande para que algunas diferencias sean visibles.

## Objetivos

Al finalizar deberías poder:

1. Distinguir `Series` y `DataFrame`.
2. Inspeccionar estructura, índices y dtypes.
3. Detectar tipos inadecuados.
4. Medir memoria global y por columna.
5. Optimizar columnas categóricas.
6. Entender el coste de `object`.
7. Seleccionar correctamente con `[]`, `.loc` e `.iloc`.
8. Diferenciar selección por etiqueta y posición.
9. Detectar chained indexing.
10. Evitar asignaciones ambiguas.
11. Comprender copias y objetos derivados.
12. Construir filtros booleanos legibles.
13. Comparar estrategias mediante benchmarks.
14. Validar que una optimización conserva el resultado.

## Cómo trabajar

**Solución:** contiene la misma parte guiada y doce ejercicios desarrollados con comprobaciones de valores, índices, rangos, filtros y asignaciones.

**Entorno:** Python, NumPy, Pandas y Matplotlib. Datos sintéticos autocontenidos de 200.000 filas. Los tiempos y la memoria dependen del entorno; las advertencias de asignación dependen de la versión y el modo de copia de Pandas.

La auditoría y los ejercicios de categoría se centran en object según el enunciado. Si tu entorno utiliza string, revisa esa representación también. El docente seleccionará los ejercicios de aula y las ampliaciones.


## 1. Preparación

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Pandas:", pd.__version__)

## 2. Crear un dataset de trabajo

In [ ]:
N = 200_000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, N + 1),
    "cliente_id": rng.integers(1, 30_001, N),
    "canal": rng.choice(
        ["web", "tienda", "partner", "marketplace"],
        N,
        p=[0.45, 0.30, 0.15, 0.10]
    ),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        N
    ),
    "unidades": rng.integers(1, 10, N),
    "precio": rng.uniform(5, 250, N).round(2),
    "descuento": rng.choice(
        [0.0, 0.05, 0.10, 0.20],
        N,
        p=[0.55, 0.20, 0.18, 0.07]
    ),
    "fecha": pd.date_range(
        "2025-01-01",
        periods=N,
        freq="min"
    )
})

ventas["importe"] = (
    ventas["unidades"]
    * ventas["precio"]
    * (1 - ventas["descuento"])
)

ventas.head()

### Interpretación

Usaremos 200.000 observaciones. No es un dataset masivo, pero sí suficientemente grande para estudiar memoria y rendimiento sin depender de ficheros externos.

## 3. Series frente a DataFrame

In [ ]:
columna = ventas["precio"]
tabla = ventas[["precio"]]

print(type(columna))
print("Shape Series:", columna.shape)

print()
print(type(tabla))
print("Shape DataFrame:", tabla.shape)

Aunque contienen los mismos valores, no son el mismo objeto:

- `ventas["precio"]` produce una `Series`;
- `ventas[["precio"]]` mantiene estructura bidimensional y produce un `DataFrame`.

Esta diferencia importa al encadenar operaciones o diseñar funciones.

## 4. Inspección estructural antes de transformar

In [ ]:
print("Shape:", ventas.shape)
print("Índice:", type(ventas.index).__name__)
print()
print(ventas.dtypes)

### 4.1 `info()`

In [ ]:
ventas.info(memory_usage="deep")

`info()` ofrece una primera auditoría: número de filas, columnas, no nulos, tipos y memoria aproximada.

## 5. Memoria por columna

In [ ]:
memoria_columnas = (
    ventas.memory_usage(deep=True)
    .sort_values(ascending=False)
    .rename("bytes")
    .to_frame()
)

memoria_columnas["MB"] = memoria_columnas["bytes"] / 1024**2
memoria_columnas

### 5.1 Visualizar qué columnas consumen más

In [ ]:
mem_plot = memoria_columnas.drop(index="Index", errors="ignore")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mem_plot.index, mem_plot["MB"])
ax.set(
    title="Memoria profunda por columna",
    xlabel="MB",
    ylabel="Columna"
)
plt.show()

### Interpretación

Las columnas de texto repetitivo suelen consumir bastante memoria cuando permanecen como `object`. No optimizaremos todavía: primero hemos cuantificado el problema.

## 6. Detectar cardinalidad

In [ ]:
cardinalidad = pd.DataFrame({
    "dtype": ventas.dtypes.astype(str),
    "n_unicos": ventas.nunique(dropna=False),
    "pct_unicos": ventas.nunique(dropna=False) / len(ventas) * 100
})

cardinalidad

Una columna textual con muy pocos valores distintos respecto al número de filas es candidata a representación categórica.

## 7. Optimizar categorías

In [ ]:
ventas_opt = ventas.copy()

for columna in ["canal", "region"]:
    ventas_opt[columna] = ventas_opt[columna].astype("category")

print(ventas_opt.dtypes)

### 7.1 Medir antes y después

In [ ]:
mem_antes = ventas.memory_usage(deep=True).sum() / 1024**2
mem_despues = ventas_opt.memory_usage(deep=True).sum() / 1024**2

comparacion_memoria = pd.DataFrame({
    "version": ["object", "category"],
    "memoria_MB": [mem_antes, mem_despues]
})

comparacion_memoria["reduccion_pct"] = (
    1 - comparacion_memoria["memoria_MB"] / mem_antes
) * 100

comparacion_memoria

### 7.2 Visualizar la reducción

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    comparacion_memoria["version"],
    comparacion_memoria["memoria_MB"]
)
ax.set(
    title="Memoria antes y después de categorizar",
    xlabel="Representación",
    ylabel="Memoria total (MB)"
)
plt.show()

### Interpretación

La optimización no cambia el significado de `canal` o `region`; cambia su representación interna. En columnas de baja cardinalidad puede reducir notablemente la memoria.

## 8. Validar que la optimización conserva valores

In [ ]:
for columna in ["canal", "region"]:
    iguales = (
        ventas[columna].astype(str).to_numpy()
        == ventas_opt[columna].astype(str).to_numpy()
    ).all()

    print(columna, "->", iguales)

Nunca deberíamos considerar válida una optimización únicamente porque consume menos. También debemos comprobar que conserva la información esperada.

## 9. Tipos numéricos y rango real

In [ ]:
ventas[["venta_id", "cliente_id", "unidades", "precio", "descuento"]].dtypes

### 9.1 Inspeccionar rangos

In [ ]:
ventas[[
    "venta_id",
    "cliente_id",
    "unidades",
    "precio",
    "descuento"
]].agg(["min", "max"])

Los rangos ayudan a decidir si un dtype es sobredimensionado. Optimizar enteros o floats requiere más cuidado que convertir categorías, porque puede afectar precisión o rango.

## 10. Downcasting controlado

In [ ]:
ventas_down = ventas_opt.copy()

ventas_down["venta_id"] = pd.to_numeric(
    ventas_down["venta_id"],
    downcast="unsigned"
)

ventas_down["cliente_id"] = pd.to_numeric(
    ventas_down["cliente_id"],
    downcast="unsigned"
)

ventas_down["unidades"] = pd.to_numeric(
    ventas_down["unidades"],
    downcast="unsigned"
)

ventas_down[["venta_id", "cliente_id", "unidades"]].dtypes

### 10.1 Volver a medir

In [ ]:
versiones_memoria = pd.DataFrame({
    "version": [
        "Original",
        "Categorías",
        "Categorías + downcast"
    ],
    "MB": [
        ventas.memory_usage(deep=True).sum() / 1024**2,
        ventas_opt.memory_usage(deep=True).sum() / 1024**2,
        ventas_down.memory_usage(deep=True).sum() / 1024**2
    ]
})

versiones_memoria["ahorro_vs_original_pct"] = (
    1 - versiones_memoria["MB"] / versiones_memoria.loc[0, "MB"]
) * 100

versiones_memoria

### 10.2 Gráfico acumulado de optimización

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(versiones_memoria["version"], versiones_memoria["MB"])
ax.set(
    title="Impacto acumulado de la optimización de dtypes",
    ylabel="Memoria (MB)"
)
ax.tick_params(axis="x", rotation=20)
plt.show()

### Interpretación

Optimizar dtypes debe ser una decisión basada en:

1. rango;
2. precisión necesaria;
3. memoria;
4. compatibilidad con operaciones posteriores.

No debe hacerse de forma ciega.

# 11. Selección con `[]`

In [ ]:
ventas["precio"].head()

In [ ]:
ventas[["precio", "unidades", "importe"]].head()

`[]` es cómodo para seleccionar columnas, pero cuando queremos combinar filas y columnas de forma explícita conviene utilizar `.loc` o `.iloc`.

# 12. `.loc`: selección por etiqueta

In [ ]:
ventas.loc[
    0:4,
    ["venta_id", "canal", "importe"]
]

Con `.loc`, tanto filas como columnas se expresan mediante **etiquetas**.

Atención: en un `RangeIndex`, `0:4` incluye la etiqueta `4`.

# 13. `.iloc`: selección por posición

In [ ]:
ventas.iloc[
    0:5,
    [0, 2, 7]
]

`.iloc` trabaja con posiciones enteras y sigue la semántica habitual de slicing de Python: el extremo final no se incluye.

# 14. Etiqueta no es posición

In [ ]:
ejemplo_indice = pd.DataFrame(
    {"valor": [10, 20, 30, 40]},
    index=[100, 200, 300, 400]
)

ejemplo_indice

In [ ]:
print("loc[200]:")
display(ejemplo_indice.loc[200])

print("iloc[1]:")
display(ejemplo_indice.iloc[1])

Ambas expresiones llegan a la misma fila en este caso, pero por mecanismos diferentes:

- `.loc[200]`: etiqueta 200;
- `.iloc[1]`: segunda posición.

# 15. Provocar un error de selección

In [ ]:
try:
    ejemplo_indice.loc[1]
except KeyError as error:
    print(type(error).__name__, "->", error)

La etiqueta `1` no existe, aunque sí exista una segunda posición. Confundir posición y etiqueta puede producir errores o, peor, seleccionar datos incorrectos.

# 16. Filtros booleanos: empezar con uno simple

In [ ]:
ventas_web = ventas.loc[
    ventas["canal"] == "web"
]

ventas_web.head()

## 16.1 Varias condiciones

In [ ]:
ventas_interes = ventas.loc[
    (ventas["canal"] == "web")
    & (ventas["importe"] >= 1000)
    & (ventas["descuento"] <= 0.10),
    ["venta_id", "canal", "importe", "descuento"]
]

ventas_interes.head()

### Importante

Cada condición se encierra entre paréntesis y se combinan con:

- `&` para AND;
- `|` para OR;
- `~` para NOT.

No se utilizan `and` y `or` sobre Series.

# 17. Mostrar el error con `and`

In [ ]:
try:
    mascara_mal = (
        (ventas["canal"] == "web")
        and
        (ventas["importe"] > 1000)
    )
except ValueError as error:
    print(type(error).__name__, "->", error)

## 17.1 Corrección

In [ ]:
mascara_correcta = (
    (ventas["canal"] == "web")
    &
    (ventas["importe"] > 1000)
)

print("Filas seleccionadas:", mascara_correcta.sum())

# 18. Máscaras con nombre para mejorar legibilidad

In [ ]:
es_web = ventas["canal"].eq("web")
es_venta_alta = ventas["importe"].ge(1000)
descuento_bajo = ventas["descuento"].le(0.10)

seleccion = ventas.loc[
    es_web & es_venta_alta & descuento_bajo
]

seleccion.head()

Nombrar máscaras puede ser preferible cuando una regla de selección tiene significado de negocio. Facilita lectura, depuración y reutilización.

# 19. `isin()` frente a cadenas de OR

In [ ]:
regiones_objetivo = ["Norte", "Este", "Centro"]

filtro_isin = ventas["region"].isin(regiones_objetivo)

ventas.loc[
    filtro_isin,
    ["venta_id", "region", "importe"]
].head()

`isin()` suele expresar con más claridad la pertenencia a un conjunto que una larga secuencia de comparaciones con `|`.

# 20. `between()`

In [ ]:
ventas.loc[
    ventas["importe"].between(500, 1000),
    ["venta_id", "importe"]
].head()

# 21. Seleccionar solo lo necesario

In [ ]:
columnas_analisis = [
    "venta_id",
    "canal",
    "region",
    "importe"
]

ventas_reducidas = ventas.loc[
    :,
    columnas_analisis
]

ventas_reducidas.head()

### 21.1 Medir memoria

In [ ]:
mem_comparacion = pd.DataFrame({
    "objeto": ["DataFrame completo", "Columnas necesarias"],
    "MB": [
        ventas.memory_usage(deep=True).sum() / 1024**2,
        ventas_reducidas.memory_usage(deep=True).sum() / 1024**2
    ]
})

mem_comparacion

### 21.2 Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(mem_comparacion["objeto"], mem_comparacion["MB"])
ax.set(
    title="Memoria según columnas conservadas",
    ylabel="MB"
)
ax.tick_params(axis="x", rotation=15)
plt.show()

### Interpretación

Reducir columnas temprano puede disminuir la memoria de los objetos intermedios. En pipelines grandes, seleccionar solo lo necesario también mejora claridad.

# 22. Chained indexing: primero la mala práctica

In [ ]:
ejemplo_chained = ventas.head(20).copy()

subset = ejemplo_chained[
    ejemplo_chained["canal"] == "web"
]

subset["segmento_importe"] = "web"

subset.head()

### ¿Por qué es problemático?

Expresiones del estilo:

```python
df[condicion]["columna"] = valor
```

mezclan selección y asignación en pasos cuya semántica puede ser ambigua. Dependiendo del objeto obtenido, podemos estar trabajando sobre una copia o un objeto derivado.

La alternativa explícita es `.loc`.

## 22.1 Asignación explícita con `.loc`

In [ ]:
ejemplo_loc = ventas.head(20).copy()

mascara = ejemplo_loc["canal"].eq("web")

ejemplo_loc.loc[
    mascara,
    "segmento_importe"
] = "web"

ejemplo_loc.head()

### Interpretación

`.loc[filas, columna] = valor` expresa en una única operación qué filas y qué columna queremos modificar.

# 23. Comprobar el objeto original

In [ ]:
original = ventas.head(10).copy()
derivado = original.loc[
    original["canal"] == "web"
].copy()

derivado["precio"] = 0

print("DERIVADO")
display(derivado[["venta_id", "precio"]])

print("ORIGINAL")
display(original[["venta_id", "precio"]])

El `.copy()` explícito comunica que queremos un objeto independiente antes de modificarlo. Esto evita depender de detalles internos sobre vistas y copias.

# 24. `.query()` como alternativa expresiva

In [ ]:
resultado_query = ventas.query(
    "canal == 'web' and importe >= 1000 and descuento <= 0.10"
)

resultado_query.head()

`.query()` puede ser muy legible en expresiones sencillas. No es automáticamente mejor ni más rápido: debe elegirse por claridad y medirse cuando el rendimiento sea relevante.

# 25. Benchmark: máscara booleana vs query

In [ ]:
def medir(funcion, repeticiones=7):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - inicio)

    return np.array(tiempos)


def filtro_booleano():
    return ventas.loc[
        (ventas["canal"] == "web")
        & (ventas["importe"] >= 1000)
        & (ventas["descuento"] <= 0.10)
    ]


def filtro_query():
    return ventas.query(
        "canal == 'web' and importe >= 1000 and descuento <= 0.10"
    )


t_bool = medir(filtro_booleano)
t_query = medir(filtro_query)

benchmark = pd.DataFrame({
    "metodo": ["Máscara booleana", "query"],
    "media_ms": [
        t_bool.mean() * 1000,
        t_query.mean() * 1000
    ],
    "std_ms": [
        t_bool.std() * 1000,
        t_query.std() * 1000
    ]
})

benchmark

## 25.1 Visualizar benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    benchmark["metodo"],
    benchmark["media_ms"],
    yerr=benchmark["std_ms"],
    capsize=4
)

ax.set(
    title="Tiempo medio de filtrado",
    ylabel="Milisegundos"
)

plt.show()

### Interpretación

El resultado depende del tamaño, expresión, versión de Pandas y hardware. La conclusión metodológica es más importante que quién gane una ejecución concreta:

> **medir antes de afirmar que una alternativa es más rápida.**

# 26. Validar equivalencia de ambos filtros

In [ ]:
a = filtro_booleano().sort_index()
b = filtro_query().sort_index()

print("Mismo shape:", a.shape == b.shape)
print("Mismos índices:", a.index.equals(b.index))

Una comparación de rendimiento solo tiene sentido si las alternativas producen el mismo resultado lógico.

# 27. Acceso escalar: `.at` y `.iat`

In [ ]:
print(".at:", ventas.at[0, "precio"])
print(".iat:", ventas.iat[0, ventas.columns.get_loc("precio")])

- `.at`: acceso escalar por etiqueta.
- `.iat`: acceso escalar por posición.

Son útiles cuando realmente necesitamos un único valor.

# 28. Benchmark escalar

In [ ]:
fila = 100_000
pos_precio = ventas.columns.get_loc("precio")

def acceso_loc():
    return ventas.loc[fila, "precio"]

def acceso_at():
    return ventas.at[fila, "precio"]

def acceso_iloc():
    return ventas.iloc[fila, pos_precio]

def acceso_iat():
    return ventas.iat[fila, pos_precio]

metodos = {
    "loc": acceso_loc,
    "at": acceso_at,
    "iloc": acceso_iloc,
    "iat": acceso_iat
}

resultados = []

for nombre, funcion in metodos.items():
    tiempos = medir(funcion, repeticiones=200)
    resultados.append({
        "metodo": nombre,
        "media_us": tiempos.mean() * 1_000_000
    })

benchmark_escalar = pd.DataFrame(resultados)
benchmark_escalar

## 28.1 Visualizar acceso escalar

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    benchmark_escalar["metodo"],
    benchmark_escalar["media_us"]
)
ax.set(
    title="Acceso a un único valor",
    xlabel="Método",
    ylabel="Tiempo medio (µs)"
)
plt.show()

### Interpretación

`.at` y `.iat` comunican explícitamente que se desea acceso escalar. Las diferencias absolutas suelen ser pequeñas, por lo que no deben convertirse en microoptimizaciones prematuras.

# 29. Ordenar antes de seleccionar top-N

In [ ]:
top_ventas = (
    ventas.loc[
        :,
        ["venta_id", "canal", "importe"]
    ]
    .sort_values(
        "importe",
        ascending=False
    )
    .head(10)
)

top_ventas

## 29.1 Visualizar las operaciones de mayor importe

In [ ]:
top_plot = top_ventas.sort_values("importe")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    top_plot["venta_id"].astype(str),
    top_plot["importe"]
)
ax.set(
    title="Top 10 operaciones por importe",
    xlabel="Importe",
    ylabel="venta_id"
)
plt.show()

### Interpretación

La selección de columnas se realiza antes de construir el objeto final que utilizaremos para comunicar el resultado.

# 30. Filtrar fechas correctamente

In [ ]:
inicio = pd.Timestamp("2025-02-01")
fin = pd.Timestamp("2025-02-28 23:59:59")

ventas_febrero = ventas.loc[
    ventas["fecha"].between(inicio, fin)
]

ventas_febrero[["fecha", "importe"]].head()

## 30.1 Validar el rango

In [ ]:
ventas_febrero["fecha"].agg(["min", "max"])

# 31. Selección de tipos con `select_dtypes`

In [ ]:
numericas = ventas.select_dtypes(include="number")
categoricas = ventas.select_dtypes(include=["object", "category"])

print("Numéricas:", list(numericas.columns))
print("Texto/categoría:", list(categoricas.columns))

Seleccionar columnas por dtype resulta útil para construir pipelines genéricos, aunque siempre debemos verificar que la semántica de las variables coincida con su dtype.

# 32. Un entero no siempre es una variable cuantitativa

In [ ]:
print("cliente_id dtype:", ventas["cliente_id"].dtype)
print("Media de cliente_id:", ventas["cliente_id"].mean())

La media de un identificador es computable, pero normalmente carece de interpretación.

**dtype técnico ≠ significado estadístico.**

Este principio será importante en análisis y feature engineering.

# 33. Función de auditoría de DataFrame

In [ ]:
def auditar_dataframe(datos: pd.DataFrame) -> pd.DataFrame:
    memoria = datos.memory_usage(deep=True)

    auditoria = pd.DataFrame({
        "dtype": datos.dtypes.astype(str),
        "nulos": datos.isna().sum(),
        "n_unicos": datos.nunique(dropna=False),
        "memoria_MB": memoria.drop("Index", errors="ignore") / 1024**2
    })

    auditoria["pct_nulos"] = (
        auditoria["nulos"] / len(datos) * 100
    )

    auditoria["pct_unicos"] = (
        auditoria["n_unicos"] / len(datos) * 100
    )

    return auditoria.sort_values(
        "memoria_MB",
        ascending=False
    )

auditoria = auditar_dataframe(ventas)
auditoria

## 33.1 Detectar candidatas a categoría

In [ ]:
candidatas_categoria = auditoria.loc[
    (auditoria["dtype"] == "object")
    & (auditoria["pct_unicos"] < 5)
]

candidatas_categoria

### Interpretación

La auditoría convierte una recomendación general —“mira los tipos”— en un diagnóstico reproducible y cuantificable.

# 34. Mini-práctica: optimizar y validar

In [ ]:
ventas_practica = ventas.copy()

memoria_inicial = (
    ventas_practica.memory_usage(deep=True).sum()
    / 1024**2
)

for columna in ["canal", "region"]:
    ventas_practica[columna] = (
        ventas_practica[columna]
        .astype("category")
    )

for columna in ["venta_id", "cliente_id", "unidades"]:
    ventas_practica[columna] = pd.to_numeric(
        ventas_practica[columna],
        downcast="unsigned"
    )

memoria_final = (
    ventas_practica.memory_usage(deep=True).sum()
    / 1024**2
)

print("Memoria inicial:", round(memoria_inicial, 2), "MB")
print("Memoria final:", round(memoria_final, 2), "MB")
print(
    "Reducción:",
    round((1 - memoria_final / memoria_inicial) * 100, 2),
    "%"
)

## 34.1 Validación funcional

In [ ]:
validaciones = {
    "mismas_filas": len(ventas) == len(ventas_practica),
    "mismos_importes": np.allclose(
        ventas["importe"],
        ventas_practica["importe"]
    ),
    "mismos_canales": (
        ventas["canal"].astype(str).to_numpy()
        ==
        ventas_practica["canal"].astype(str).to_numpy()
    ).all()
}

validaciones

## 34.2 Comparación gráfica final

In [ ]:
comparacion_final = pd.DataFrame({
    "version": ["Original", "Optimizada"],
    "MB": [memoria_inicial, memoria_final]
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    comparacion_final["version"],
    comparacion_final["MB"]
)
ax.set(
    title="Resultado de la optimización",
    ylabel="Memoria (MB)"
)
plt.show()

### Interpretación final de la práctica

La optimización se acepta porque cumple dos condiciones:

1. reduce recursos;
2. conserva los resultados relevantes.

Este doble criterio será recurrente en los siguientes laboratorios de rendimiento.

# 35. Checklist de selección eficiente

- [ ] ¿He inspeccionado `shape`, índice y dtypes?
- [ ] ¿He medido memoria con `deep=True`?
- [ ] ¿Hay strings repetitivos candidatos a `category`?
- [ ] ¿El rango permite downcasting?
- [ ] ¿He validado la precisión después de optimizar?
- [ ] ¿Uso `.loc` cuando selecciono por etiqueta?
- [ ] ¿Uso `.iloc` cuando selecciono por posición?
- [ ] ¿Evito chained indexing?
- [ ] ¿Uso `.copy()` cuando quiero independencia explícita?
- [ ] ¿Las máscaras complejas tienen nombres legibles?
- [ ] ¿Uso `isin()` para pertenencia?
- [ ] ¿Selecciono solo columnas necesarias?
- [ ] ¿Mido antes de afirmar que algo es más rápido?
- [ ] ¿Valido equivalencia entre alternativas?

# 36. Ejercicios finales


## Ejercicio 1 — Series y DataFrames

Demuestra cinco selecciones sobre ventas. Documenta el tipo, shape y significado de cada resultado. Incluye Series, DataFrame de una columna, varias columnas, fila y escalar.


In [ ]:
selecciones_practica = {
    "columna Series": ventas["precio"],
    "columna DataFrame": ventas[["precio"]],
    "varias columnas": ventas[["precio", "unidades"]],
    "fila Series": ventas.iloc[0],
    "valor escalar": ventas.at[ventas.index[0], "precio"],
}
inspeccion_selecciones = pd.DataFrame([
    {"seleccion": nombre, "tipo": type(objeto).__name__, "shape": np.shape(objeto)}
    for nombre, objeto in selecciones_practica.items()])
assert selecciones_practica["columna Series"].ndim == 1
assert selecciones_practica["columna DataFrame"].ndim == 2
np.testing.assert_array_equal(ventas["precio"].to_numpy(), ventas[["precio"]].to_numpy().ravel())
print(inspeccion_selecciones)


### Solución razonada

La Series de columna y el DataFrame de una columna contienen los mismos valores, pero tienen distinta dimensionalidad y API. Una fila seleccionada como Series puede combinar tipos de columnas; un escalar no tiene ejes tabulares.


## Ejercicio 2 — Auditoría

Construye una tabla por columna con dtype, nulos, cardinalidad y memoria profunda. Define qué ocurre con una tabla vacía y cómo contabilizar el índice.


In [ ]:
def auditoria_practica(datos: pd.DataFrame) -> pd.DataFrame:
    """Describe columnas; memory_usage(deep=True) no mide el pico de memoria del proceso."""
    filas = len(datos)
    return pd.DataFrame({
        "dtype": datos.dtypes.astype(str),
        "nulos": datos.isna().sum(),
        "n_unicos": datos.nunique(dropna=True),
        "ratio_unicos": datos.nunique(dropna=True) / filas if filas else np.nan,
        "memoria_bytes": datos.memory_usage(index=False, deep=True),
    })

auditoria_ventas = auditoria_practica(ventas)
assert auditoria_ventas.index.equals(ventas.columns)
assert auditoria_ventas["memoria_bytes"].sum() == ventas.memory_usage(index=False, deep=True).sum()
assert auditoria_practica(ventas.iloc[:0])["ratio_unicos"].isna().all()
print(auditoria_ventas)
print("Memoria del índice:", ventas.index.memory_usage(deep=True), "bytes")


### Solución razonada

La cardinalidad excluye nulos y su proporción utiliza el número total de filas. El índice se mide por separado. La memoria atribuida al DataFrame no equivale al consumo total ni al máximo alcanzado por el proceso.


## Ejercicio 3 — Categorías

Detecta automáticamente columnas object con menos del 5 % de valores únicos. Convierte en una copia y valida que valores, nulos e índice se mantienen. No conviertas identificadores solo por su dtype.


In [ ]:
def categorizar_practica(datos: pd.DataFrame, limite: float = 0.05) -> tuple[pd.DataFrame, list[str]]:
    """Convierte columnas object con proporción de únicos menor que el límite."""
    if not 0 < limite <= 1:
        raise ValueError("limite debe estar en (0, 1].")
    salida = datos.copy()
    if datos.empty:
        return salida, []
    candidatas = [columna for columna in datos.select_dtypes(include="object").columns
                  if datos[columna].nunique(dropna=True) / len(datos) < limite]
    for columna in candidatas:
        salida[columna] = salida[columna].astype("category")
    return salida, candidatas

ventas_categorias_practica, candidatas_practica = categorizar_practica(ventas)
for columna in candidatas_practica:
    pd.testing.assert_series_equal(ventas[columna], ventas_categorias_practica[columna].astype(ventas[columna].dtype))
assert ventas_categorias_practica.index.equals(ventas.index)
assert ventas_categorias_practica.isna().equals(ventas.isna())
assert ventas["venta_id"].dtype == np.dtype("int64")
print("Columnas convertidas:", candidatas_practica)


### Solución razonada

El criterio del 5 % es una heurística, no una garantía de ahorro. category resulta especialmente útil con valores repetidos; medir memoria y revisar el significado siguen siendo necesarios. Este ejercicio detecta específicamente object; si el entorno usa string, hay que revisar también ese dtype antes de ampliar la selección.


## Ejercicio 4 — Memoria

Mide la memoria antes y después de categorías y representa ambos valores. Cuantifica el ahorro y distingue memoria del objeto de memoria máxima del proceso.


In [ ]:
bytes_original_practica = ventas.memory_usage(deep=True).sum()
bytes_categorias_practica = ventas_categorias_practica.memory_usage(deep=True).sum()
ahorro_practica = 100 * (1 - bytes_categorias_practica / bytes_original_practica)
memoria_practica = pd.DataFrame({"version": ["Original", "Categorías"],
    "MiB": [bytes_original_practica / 1024**2, bytes_categorias_practica / 1024**2]})
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(memoria_practica["version"], memoria_practica["MiB"])
ax.set(title="Memoria atribuida al DataFrame", ylabel="MiB")
plt.show()
print(memoria_practica)
print(f"Reducción medida: {ahorro_practica:.2f} %")


### Solución razonada

Las barras muestran el ahorro real en este dataset y entorno. Mantener simultáneamente el original y la copia puede elevar la memoria del proceso durante la transformación; memory_usage no mide ese pico. El ahorro solo se acepta junto a las comprobaciones de equivalencia.


## Ejercicio 5 — Downcasting

Inspecciona rango y dtype de venta_id, cliente_id y unidades. Optimiza estos enteros y comprueba igualdad exacta de valores. Explica por qué no debe convertirse un identificador a float.


In [ ]:
columnas_enteras_practica = ["venta_id", "cliente_id", "unidades"]
print(ventas_categorias_practica[columnas_enteras_practica].agg(["min", "max"]))
ventas_down_practica = ventas_categorias_practica.copy()
for columna in columnas_enteras_practica:
    if ventas_down_practica[columna].min() < 0:
        raise ValueError(f"{columna} no admite unsigned.")
    ventas_down_practica[columna] = pd.to_numeric(ventas_down_practica[columna], downcast="unsigned")
    np.testing.assert_array_equal(ventas_down_practica[columna].to_numpy(), ventas[columna].to_numpy())
assert ventas_down_practica.index.equals(ventas.index)
print(ventas_down_practica[columnas_enteras_practica].dtypes)
print("Bytes antes:", ventas_categorias_practica.memory_usage(deep=True).sum())
print("Bytes después:", ventas_down_practica.memory_usage(deep=True).sum())


### Solución razonada

El downcasting de enteros puede conservar valores exactamente cuando todos caben en el rango elegido. Los identificadores no representan una magnitud para promediar y un float puede perder precisión en enteros grandes. No se modifica el importe: reducir precisión decimal exigiría una tolerancia de negocio explícita.


## Ejercicio 6 — .loc vs .iloc

Crea un DataFrame con índice [10, 20, 30, 40]. Muestra acceso por etiqueta y posición, slices inclusivo/exclusivo y el error de una etiqueta ausente.


In [ ]:
tabla_indices_practica = pd.DataFrame({"valor": [100, 200, 300, 400]}, index=[10, 20, 30, 40])
pd.testing.assert_series_equal(tabla_indices_practica.loc[20], tabla_indices_practica.iloc[1])
pd.testing.assert_frame_equal(tabla_indices_practica.loc[10:30], tabla_indices_practica.iloc[0:3])
assert len(tabla_indices_practica.loc[10:30]) == 3
assert len(tabla_indices_practica.iloc[0:2]) == 2
try:
    tabla_indices_practica.loc[1]
except KeyError as error:
    print("Etiqueta ausente:", error)
else:
    raise AssertionError("La etiqueta 1 no debería existir.")
print(tabla_indices_practica.loc[10:30])


### Solución razonada

loc usa etiquetas y, en este índice ordenado, incluye los dos extremos del slice. iloc usa posiciones y excluye el extremo derecho. Que ambas lleguen a una misma fila no las convierte en operaciones intercambiables.


## Ejercicio 7 — Error booleano

Provoca el error de usar and entre Series y corrige con &. Usa paréntesis en cada comparación y valida las filas resultantes.


In [ ]:
try:
    mascara_erronea_practica = (ventas["canal"] == "web") and (ventas["importe"] >= 1000)
except ValueError as error:
    print("Error esperado:", error)
else:
    raise AssertionError("and entre Series debería fallar.")
mascara_correcta_practica = ventas["canal"].eq("web") & ventas["importe"].ge(1000)
resultado_booleano_practica = ventas.loc[mascara_correcta_practica]
assert resultado_booleano_practica["canal"].eq("web").all()
assert resultado_booleano_practica["importe"].ge(1000).all()
print("Filas seleccionadas:", len(resultado_booleano_practica))


### Solución razonada

and intenta obtener un único valor de verdad de toda la Series. & opera elemento a elemento. Los paréntesis delimitan comparaciones cuando se utilizan operadores como == o >=.


## Ejercicio 8 — Chained indexing

Construye una asignación encadenada y compárala con una asignación explícita usando loc. Comprueba las filas que cambian y las que deben mantenerse. No presupongas que aparezca una advertencia en todas las versiones de Pandas.


In [ ]:
import warnings
base_asignacion = pd.DataFrame({"canal": ["web", "tienda", "web"], "marca": ["sin", "sin", "sin"]})
demo_ambigua = base_asignacion.copy()
with warnings.catch_warnings(record=True) as avisos_practica:
    warnings.simplefilter("always")
    demo_ambigua[demo_ambigua["canal"].eq("web")]["marca"] = "revisar"
print("Resultado de la asignación encadenada:")
print(demo_ambigua)
print("Advertencias capturadas:", [type(aviso.message).__name__ for aviso in avisos_practica])
demo_explicita = base_asignacion.copy()
mascara_web_practica = demo_explicita["canal"].eq("web")
demo_explicita.loc[mascara_web_practica, "marca"] = "revisar"
assert demo_explicita.loc[mascara_web_practica, "marca"].eq("revisar").all()
assert demo_explicita.loc[~mascara_web_practica, "marca"].eq("sin").all()
pd.testing.assert_frame_equal(base_asignacion, pd.DataFrame({"canal": ["web", "tienda", "web"], "marca": ["sin", "sin", "sin"]}))
print("Asignación explícita comprobada:")
print(demo_explicita)


### Solución razonada

La asignación encadenada actúa sobre un resultado intermedio y no es una forma fiable de actualizar el objeto original. Su advertencia y comportamiento dependen del modo de copia y versión. loc expresa una actualización única del DataFrame seleccionado; las pruebas verifican el resultado, no la presencia de una advertencia.


## Ejercicio 9 — Máscaras con nombre

Selecciona ventas web, de regiones Norte/Este/Centro, importe >= 1000 y descuento <= 0.10. Usa cuatro máscaras descriptivas y selecciona únicamente venta_id, canal, region e importe.


In [ ]:
es_web_practica = ventas["canal"].eq("web")
es_region_objetivo_practica = ventas["region"].isin(["Norte", "Este", "Centro"])
es_importe_alto_practica = ventas["importe"].ge(1000)
es_descuento_bajo_practica = ventas["descuento"].le(0.10)
mascara_negocio_practica = es_web_practica & es_region_objetivo_practica & es_importe_alto_practica & es_descuento_bajo_practica
resultado_negocio_practica = ventas.loc[mascara_negocio_practica, ["venta_id", "canal", "region", "importe"]].copy()
assert resultado_negocio_practica["canal"].eq("web").all()
assert resultado_negocio_practica["region"].isin(["Norte", "Este", "Centro"]).all()
assert resultado_negocio_practica["importe"].ge(1000).all()
assert ventas.loc[resultado_negocio_practica.index, "descuento"].le(0.10).all()
print(resultado_negocio_practica.head())


### Solución razonada

Cada máscara comunica una regla de negocio y mantiene el índice de ventas. Reducir columnas limita el tamaño de la salida. Tener nombres descriptivos permite revisar condiciones sin convertir el filtro en una expresión difícil de leer.


## Ejercicio 10 — isin y between

Combina regiones Norte/Este y un importe entre 500 y 1000, con ambos extremos incluidos. Comprueba equivalencia con comparaciones explícitas.


In [ ]:
mascara_rango_practica = ventas["region"].isin(["Norte", "Este"]) & ventas["importe"].between(500, 1000, inclusive="both")
resultado_rango_practica = ventas.loc[mascara_rango_practica]
mascara_rango_explicita = ((ventas["region"].eq("Norte") | ventas["region"].eq("Este"))
                           & ventas["importe"].ge(500) & ventas["importe"].le(1000))
pd.testing.assert_frame_equal(resultado_rango_practica, ventas.loc[mascara_rango_explicita])
assert pd.Series([499, 500, 1000, 1001]).between(500, 1000, inclusive="both").tolist() == [False, True, True, False]
print("Pertenencia y extremos inclusivos comprobados.")


### Solución razonada

isin comunica pertenencia a un conjunto; between comunica un intervalo. Hacer explícita la inclusión de extremos evita ambigüedad. La prueba aislada con 500 y 1000 no depende de que esas cantidades aparezcan en la muestra.


## Ejercicio 11 — Benchmark

Compara máscara booleana y query sobre las mismas filas y columnas. Valida equivalencia primero. Mide diez repeticiones por método, alterna el orden y grafica media y desviación estándar.


In [ ]:
def comparar_filtros_practica(datos: pd.DataFrame, repeticiones: int = 10):
    """Compara filtros equivalentes; la generación de entrada queda fuera del tiempo."""
    if repeticiones < 2:
        raise ValueError("Se requieren al menos dos repeticiones para la dispersión.")
    def por_mascara():
        return datos.loc[datos["canal"].eq("web") & datos["importe"].ge(1000) & datos["descuento"].le(0.10)]
    def por_query():
        return datos.query("canal == 'web' and importe >= 1000 and descuento <= 0.10")
    pd.testing.assert_frame_equal(por_mascara(), por_query())
    por_mascara()
    por_query()
    medidas = []
    for repeticion in range(repeticiones):
        metodos = [("Máscara", por_mascara), ("query", por_query)]
        if repeticion % 2:
            metodos.reverse()
        for nombre, funcion in metodos:
            inicio = time.perf_counter()
            resultado = funcion()
            duracion = time.perf_counter() - inicio
            medidas.append({"repeticion": repeticion + 1, "metodo": nombre, "ms": duracion * 1000})
    tabla = pd.DataFrame(medidas)
    resumen = tabla.groupby("metodo")["ms"].agg(media="mean", desviacion="std", mediana="median").reset_index()
    return tabla, resumen

def graficar_benchmark_practica(resumen, titulo):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(resumen["metodo"], resumen["media"], yerr=resumen["desviacion"], capsize=5)
    ax.set(title=titulo, ylabel="Milisegundos (media ± desviación estándar)")
    plt.show()

tiempos_filtros_practica, resumen_tiempos_practica = comparar_filtros_practica(ventas)
assert tiempos_filtros_practica.groupby("metodo").size().eq(10).all()
graficar_benchmark_practica(resumen_tiempos_practica, "Filtros equivalentes: diez repeticiones")
print(resumen_tiempos_practica)


### Solución razonada

Las barras de error representan dispersión entre mediciones, no un intervalo de confianza. El método ganador depende del entorno y de los datos. No se fija un factor mínimo de aceleración; la corrección se comprueba independientemente del tiempo.


## Ejercicio 12 — Reto final

Utiliza las 200.000 filas de ventas: audita, detecta dtypes, mide memoria, convierte categorías y enteros, valida equivalencia, filtra, demuestra una mala asignación, corrige con loc y compara filtros. Interpreta ambos gráficos.


In [ ]:
entrada_reto_pandas = ventas.copy(deep=True)
assert len(entrada_reto_pandas) >= 100_000
original_reto_pandas = entrada_reto_pandas.copy(deep=True)
print(auditoria_practica(entrada_reto_pandas))
memoria_reto_antes = entrada_reto_pandas.memory_usage(deep=True).sum()
optimizado_reto_pandas, candidatas_reto_pandas = categorizar_practica(entrada_reto_pandas)
for columna in ["venta_id", "cliente_id", "unidades"]:
    optimizado_reto_pandas[columna] = pd.to_numeric(optimizado_reto_pandas[columna], downcast="unsigned")
for columna in entrada_reto_pandas.columns:
    restaurada = optimizado_reto_pandas[columna].astype(entrada_reto_pandas[columna].dtype)
    pd.testing.assert_series_equal(restaurada, entrada_reto_pandas[columna])
assert optimizado_reto_pandas.index.equals(entrada_reto_pandas.index)
assert optimizado_reto_pandas.isna().equals(entrada_reto_pandas.isna())
pd.testing.assert_frame_equal(entrada_reto_pandas, original_reto_pandas)
memoria_reto_despues = optimizado_reto_pandas.memory_usage(deep=True).sum()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Antes", "Después"], [memoria_reto_antes / 1024**2, memoria_reto_despues / 1024**2])
ax.set(title="Reto: memoria tras optimización", ylabel="MiB")
plt.show()
# Comparar el mismo filtro sobre ambas representaciones, preservando valores.
def seleccionar_reto(datos):
    mascara = (datos["canal"].eq("web") & datos["region"].isin(["Norte", "Este"])
               & datos["importe"].ge(1000) & datos["descuento"].le(0.10))
    return datos.loc[mascara, ["venta_id", "canal", "region", "importe"]].copy()
seleccion_original_reto = seleccionar_reto(entrada_reto_pandas)
seleccion_optimizada_reto = seleccionar_reto(optimizado_reto_pandas)
for columna in seleccion_original_reto.columns:
    pd.testing.assert_series_equal(seleccion_optimizada_reto[columna].astype(seleccion_original_reto[columna].dtype), seleccion_original_reto[columna])
# Asignación a columna nueva de texto, sin modificar categorías originales.
asignacion_reto = optimizado_reto_pandas.head(100).copy()
asignacion_reto["revision"] = "pendiente"
mascara_revision_reto = asignacion_reto["canal"].eq("web")
with warnings.catch_warnings(record=True) as avisos_reto:
    warnings.simplefilter("always")
    asignacion_reto[mascara_revision_reto]["revision"] = "revisar"
print("Asignación encadenada: no utilizar para actualizar el original.")
print(asignacion_reto["revision"].value_counts())
asignacion_reto.loc[mascara_revision_reto, "revision"] = "revisar"
assert asignacion_reto.loc[mascara_revision_reto, "revision"].eq("revisar").all()
assert asignacion_reto.loc[~mascara_revision_reto, "revision"].eq("pendiente").all()
tiempos_reto_pandas, resumen_benchmark_reto = comparar_filtros_practica(optimizado_reto_pandas)
assert tiempos_reto_pandas.groupby("metodo").size().eq(10).all()
graficar_benchmark_practica(resumen_benchmark_reto, "Reto: filtros en la tabla optimizada")
print(resumen_benchmark_reto)
print(f"Reducción de memoria: {100 * (1 - memoria_reto_despues / memoria_reto_antes):.2f} %")
print("Reto verificado: valores, nulos, índices, selección y asignaciones.")


### Solución razonada

**Memoria:** la reducción observada se debe al cambio de representación de textos repetidos y enteros con rango acotado. Los valores, nulos e índices se comprueban antes de aceptar la optimización; el importe permanece sin reducción de precisión. La comparación no mide el pico de memoria del proceso.

**Benchmark:** las barras comparan dos filtros equivalentes sobre la misma tabla optimizada; su dispersión muestra variabilidad del entorno. No prueban que una estrategia sea siempre superior.

**Asignación:** crear la nueva columna revision y actualizarla con loc evita ampliar las categorías existentes y hace explícita la intención. El original se conserva, tanto al optimizar como al trabajar sobre el subconjunto.


# 37. Preguntas de reflexión


1. ¿Una `Series` y un `DataFrame` de una columna son equivalentes?
2. ¿Por qué `object` puede consumir mucha memoria?
3. ¿Cuándo tiene sentido `category`?
4. ¿Qué riesgo tiene el downcasting?
5. ¿Qué diferencia fundamental hay entre `.loc` e `.iloc`?
6. ¿Por qué no usamos `and` entre Series booleanas?
7. ¿Qué problema intenta evitar `.loc` en las asignaciones?
8. ¿Por qué seleccionar menos columnas puede ser beneficioso?
9. ¿Una alternativa más legible es necesariamente más lenta?
10. ¿Por qué debemos validar equivalencia después de optimizar?

## Respuestas orientativas

1. Pueden contener los mismos valores, pero tienen dimensionalidad y comportamiento distintos: Series es unidimensional y DataFrame bidimensional.
2. object suele almacenar referencias a objetos Python; el texto repetido puede requerir más memoria que códigos categóricos. Debe medirse con deep=True.
3. Cuando el significado admite un conjunto de categorías y los valores se repiten lo suficiente para compensar el almacenamiento de códigos y categorías.
4. Puede provocar pérdida de precisión, desbordamiento o un rango insuficiente para datos futuros. Conviene inspeccionar rango, validar valores y acordar tolerancias para floats.
5. loc usa etiquetas; iloc usa posiciones. Los extremos de sus slices siguen reglas diferentes.
6. and busca un único valor de verdad; & combina máscaras elemento a elemento.
7. Evita una asignación encadenada a un objeto intermedio: una operación loc identifica las filas y columnas del DataFrame a actualizar.
8. Reduce el tamaño de resultados e intermediarios y limita columnas ajenas a la pregunta analítica; no elimina automáticamente el original de memoria.
9. No. La legibilidad no determina por sí sola el coste; hay que comparar alternativas equivalentes en el entorno real.
10. Ahorrar memoria o tiempo no sirve si cambia filas, valores, nulos, índices o reglas de selección. La validación fija el contrato que debe conservarse.
